# fyM1 01 — QLoRA training (T4)
**Runtime → Run all.** Trains a LoRA adapter on the private `Fan1000-Prog/fyM1-chats` data (every `*.jsonl`
there, rows `{"id", "messages"}`, plus the owner-approved `system_prompt.txt`). Each saved checkpoint is pushed
to `Fan1000-Prog/fyM1-ckpt` as `checkpoint-<step>/`; rerunning after a disconnect resumes from the newest one.
Stops early when eval loss rises 2 evals in a row (or is NaN); the best checkpoint becomes `final/`.
Writes `results/train_log.json` to `fyM1-ckpt`. Needs Colab secret `HF_TOKEN` and `src/` under `code/src/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U unsloth unsloth_zoo

In [ ]:
MODEL_ID = "google/gemma-4-12B-it"  # set by the lead's model pick
LR, LR_SCHEDULER, WARMUP, EPOCHS, MAX_LEN = 2e-4, "cosine", 10, 2, 2048
BATCH, GRAD_ACCUM, SAVE_EVERY = 2, 4, 100
LORA_R, LORA_ALPHA = 16, 16
EVAL_SHARE, SEED = 0.02, 42
CKPT_REPO, DATA_REPO = "Fan1000-Prog/fyM1-ckpt", "Fan1000-Prog/fyM1-chats"
OUT = "/content/out"

In [ ]:
import os, sys, json, time, random, shutil
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, list_repo_files

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never print it
api = HfApi()
code_dir = snapshot_download(CKPT_REPO, allow_patterns=["code/**"], local_dir="/content/fym1code")
sys.path.insert(0, f"{code_dir}/code/src")
from fym1.train.resume import latest_checkpoint
from fym1.train.chat_format import CHAT_TEMPLATE_KWARGS, training_text, turn_markers
from fym1.train.supervise import best_eval, eval_losses, should_stop

In [ ]:
import torch
from unsloth import FastModel

model, processor = FastModel.from_pretrained(MODEL_ID, max_seq_length=MAX_LEN, load_in_4bit=True, full_finetuning=False)
tok = getattr(processor, "tokenizer", processor)
lora = dict(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=SEED,
            use_gradient_checkpointing="unsloth")
if hasattr(model.config, "vision_config"):
    model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                     finetune_attention_modules=True, finetune_mlp_modules=True, **lora)
else:
    model = FastModel.get_peft_model(model, target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                                            "gate_proj", "up_proj", "down_proj"], **lora)
compute_dtype = next(p.dtype for p in model.parameters() if p.is_floating_point())
print("model dtype", compute_dtype, "| vram GB", round(torch.cuda.memory_allocated() / 2**30, 2))

In [ ]:
from datasets import Dataset


def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt,
                                   **CHAT_TEMPLATE_KWARGS)


files = [f for f in list_repo_files(DATA_REPO, repo_type="dataset") if f.endswith(".jsonl")]
system_prompt = open(hf_hub_download(DATA_REPO, "system_prompt.txt", repo_type="dataset")).read().strip()
rows = []
for f in files:
    rows += [json.loads(l) for l in open(hf_hub_download(DATA_REPO, f, repo_type="dataset")) if l.strip()]
random.Random(SEED).shuffle(rows)
texts = [training_text(render, [{"role": "system", "content": system_prompt}] + r["messages"]) for r in rows]
n_eval = max(1, round(len(texts) * EVAL_SHARE))
train_ds, eval_ds = Dataset.from_dict({"text": texts[n_eval:]}), Dataset.from_dict({"text": texts[:n_eval]})
lengths = [len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts]
DATA_STATS = {"files": files, "examples": len(texts), "train": len(train_ds), "eval": n_eval,
              "tokens": sum(lengths), "over_max_len": sum(l > MAX_LEN for l in lengths)}
print(DATA_STATS)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only


class PushAndSupervise(TrainerCallback):
    """Push every checkpoint to the Hub; stop when eval loss rises 2 evals in a row or is NaN."""

    def on_save(self, args, state, control, **kw):
        name = f"checkpoint-{state.global_step}"
        api.upload_folder(folder_path=f"{args.output_dir}/{name}", path_in_repo=name, repo_id=CKPT_REPO,
                          commit_message=f"train: {name}")

    def on_evaluate(self, args, state, control, **kw):
        if should_stop([l for _, l in eval_losses(state.log_history)]):
            print("STOP: eval loss rose 2 evals in a row (or NaN)")
            control.should_training_stop = True
            control.should_save = True


tok.padding_side = "right"
trainer = SFTTrainer(
    model=model, processing_class=tok, train_dataset=train_ds, eval_dataset=eval_ds,
    callbacks=[PushAndSupervise()],
    args=SFTConfig(output_dir=OUT, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
                   gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS, learning_rate=LR,
                   lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP, logging_steps=10, optim="adamw_8bit",
                   fp16=compute_dtype == torch.float16, bf16=False, max_length=MAX_LEN, dataset_text_field="text",
                   packing=False, eval_strategy="steps", eval_steps=SAVE_EVERY, save_strategy="steps",
                   save_steps=SAVE_EVERY, save_total_limit=2, report_to="none", seed=SEED))
instruction_part, response_part = turn_markers(render)
trainer = train_on_responses_only(trainer, instruction_part=instruction_part, response_part=response_part)

resume = latest_checkpoint(CKPT_REPO)
if resume:
    snapshot_download(CKPT_REPO, allow_patterns=[f"{resume}/*"], local_dir=OUT)
    base = json.load(open(f"{OUT}/{resume}/adapter_config.json")).get("base_model_name_or_path", "")
    assert MODEL_ID.split("/")[-1].lower() in base.lower(), f"{resume} was trained on {base}, not {MODEL_ID}"
print("RESUME FROM", resume or "scratch")
t0 = time.time()
trainer.train(resume_from_checkpoint=f"{OUT}/{resume}" if resume else None)
session_s = round(time.time() - t0)
trainer.save_model(f"{OUT}/last")  # final weights, also when stopped early; not a resumable checkpoint-<step>
api.upload_folder(folder_path=f"{OUT}/last", path_in_repo="last", repo_id=CKPT_REPO, commit_message="train: last")
print("TRAIN_DONE step", trainer.state.global_step, "session seconds", session_s)

In [ ]:
history = trainer.state.log_history
best = best_eval(history)
final_ckpt = f"checkpoint-{best[0]}" if best else "last"
local = snapshot_download(CKPT_REPO, allow_patterns=[f"{final_ckpt}/*"], local_dir="/content/final_src")
adapter_files = [f for f in os.listdir(f"{local}/{final_ckpt}") if f.startswith("adapter_")]
assert adapter_files, f"no adapter files in {final_ckpt}"
os.makedirs("/content/final", exist_ok=True)
for f in adapter_files:
    shutil.copy(f"{local}/{final_ckpt}/{f}", f"/content/final/{f}")
tok.save_pretrained("/content/final")
api.upload_folder(folder_path="/content/final", path_in_repo="final", repo_id=CKPT_REPO,
                  commit_message=f"final adapter = {final_ckpt}")

try:
    prev = json.load(open(hf_hub_download(CKPT_REPO, "results/train_log.json", force_download=True)))
except Exception:
    prev = {}
losses = [h["loss"] for h in history if "loss" in h]
log = {
    "gpu": torch.cuda.get_device_name(0), "model": MODEL_ID, "steps": trainer.state.global_step,
    "epochs": EPOCHS, "epochs_done": round(trainer.state.epoch or 0, 3),
    "wall_time_s": prev.get("wall_time_s", 0) + session_s, "sessions": prev.get("sessions", 0) + 1,
    "final_train_loss": losses[-1] if losses else None, "best_eval_loss": best[1] if best else None,
    "best_step": best[0] if best else None, "final_adapter": final_ckpt, "data": DATA_STATS,
    "config": {"lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "lr_scheduler": LR_SCHEDULER,
               "warmup_steps": WARMUP, "max_seq_len": MAX_LEN, "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
               "effective_batch": BATCH * GRAD_ACCUM, "compute_dtype": str(compute_dtype)},
    "loss_curve": [{"step": h["step"], "loss": h["loss"]} for h in history if "loss" in h],
    "eval_curve": [{"step": s, "eval_loss": l} for s, l in eval_losses(history)],
    "decisions": prev.get("decisions", []),
}
with open("/content/train_log.json", "w") as f:
    json.dump(log, f, indent=2)
api.upload_file(path_or_fileobj="/content/train_log.json", path_in_repo="results/train_log.json", repo_id=CKPT_REPO,
                commit_message="train log")
print("TRAIN_LOG", json.dumps({k: v for k, v in log.items() if not k.endswith("_curve")}))